<a href="https://colab.research.google.com/github/RDGopal/IB9AU-2026/blob/main/FT4_Fine_tuning_with_Image_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning with Image Data
**Fair warning: Even with GPU usage, this code will take a long while to run.**

This notebook is split into two clearly marked sections:

1. **Part 1 – Fine-Tuning** (Cells below until the divider): Downloads the dataset, prepares it, fine-tunes Qwen2-VL with LoRA, and saves the model to Google Drive.
2. **Part 2 – Inference** (Cells after the divider): Loads the saved model and evaluates it on a test set. If you've already completed fine-tuning, you can skip directly to Part 2.

---
## ⚙️ PART 1: Fine-Tuning
> Run all cells in this section to train and save the model.
> ⚠️ **Skip this section entirely if you have already trained and saved the model.**

### Step 1.1 – Install Dependencies

We pin specific library versions for reproducibility. Installing from git ensures we have Qwen2-VL support, which may not yet be in a stable release of `transformers`.

> ⚠️ **Note:** You will need a Kaggle API key (`kaggle.json`) configured in your Colab environment to download the dataset in the next step. Go to your [Kaggle account settings](https://www.kaggle.com/settings) → API → Create New Token, then upload the file when prompted.

In [ ]:
# Colab pre-installs a nightly transformers (5.x dev) which requires accelerate>=1.1.0.
# Rather than fight the system install, we work with it and just ensure
# accelerate and other dependencies are at compatible versions.

import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install",
    "accelerate>=1.1.0",
    "qwen-vl-utils",
    "bitsandbytes>=0.43.0",
    "peft>=0.13.0",
    "datasets>=3.0.0",
    "-q"], check=True)

# Verify versions
import importlib
import transformers, accelerate, bitsandbytes
print(f"transformers: {transformers.__version__}")
print(f"accelerate:   {accelerate.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print()
print("✅ Installation complete.")
print("⚠️  Go to Runtime → Restart session, then continue from Step 1.2.")

### Step 1.2 – Download and Inspect the Dataset

We use the `car-damage-severity` dataset from Kaggle Hub. It contains images of damaged cars with COCO-format annotations for damage categories (e.g., scratches, dents, cracks).

> 💡 **Tip:** If prompted for your Kaggle credentials, upload your `kaggle.json` file or set the `KAGGLE_USERNAME` and `KAGGLE_KEY` environment variables.

In [ ]:
import kagglehub
dataset_path = kagglehub.dataset_download("sujithkumarmanick/car-damage-severity")
print(f"Dataset downloaded to: {dataset_path}")

After downloading, let's inspect the directory structure to understand how images and annotation files are organised.

In [ ]:
import os

print(f"Contents of {dataset_path}:")
for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:  # Limit depth for readability
        subindent = ' ' * 2 * (level + 1)
        for f in files:
            print(f'{subindent}{f}')

Let's display a few sample images to confirm the download was successful and to get a feel for the data.

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

valid_images_path = os.path.join(dataset_path, 'Car Damage Severity Detection-CarDD.v3i.coco', 'valid')
image_files = [f for f in os.listdir(valid_images_path) if f.endswith(('.jpg', '.jpeg', '.png'))]

num_images_to_display = 3
plt.figure(figsize=(15, 5))
for i, image_name in enumerate(image_files[:num_images_to_display]):
    image_path = os.path.join(valid_images_path, image_name)
    img = Image.open(image_path)
    plt.subplot(1, num_images_to_display, i + 1)
    plt.imshow(img)
    plt.title(f"Image {i+1}")
    plt.axis('off')
plt.tight_layout()
plt.show()

### Step 1.3 – Load and Explore Annotations

The dataset uses COCO JSON format. We load the training annotations to understand the label structure: which damage categories exist and how many images and annotations there are.

In [ ]:
import json
import pandas as pd

annotation_file_path = os.path.join(
    dataset_path,
    'Car Damage Severity Detection-CarDD.v3i.coco',
    'train_annotations.coco.json'
)

with open(annotation_file_path, 'r') as f:
    annotations = json.load(f)

print(f"Number of images:      {len(annotations['images'])}")
print(f"Number of annotations: {len(annotations['annotations'])}")
print(f"Number of categories:  {len(annotations['categories'])}")

print("\nFirst 5 annotations:")
display(pd.DataFrame(annotations['annotations']).head())

print("\nCategories:")
display(pd.DataFrame(annotations['categories']))

### Step 1.4 – Convert Annotations to a DataFrame

We merge the annotation data with image filenames and category names to get a clean tabular view of the labels.

In [ ]:
import pandas as pd

images_df = pd.DataFrame(annotations['images'])
annotations_df = pd.DataFrame(annotations['annotations'])
categories_df = pd.DataFrame(annotations['categories'])

# Rename to avoid column conflicts on merge
categories_df = categories_df.rename(columns={'id': 'category_id', 'name': 'category_name'})

# Join annotations with categories, then with image filenames
merged_annotations = pd.merge(
    annotations_df,
    categories_df[['category_id', 'category_name']],
    on='category_id',
    how='left'
)
final_df = pd.merge(
    images_df[['id', 'file_name']],
    merged_annotations[['image_id', 'category_name']],
    left_on='id',
    right_on='image_id',
    how='left'
)

print(f"Total annotation rows: {len(final_df)}")
print("\nSample (first 10 rows):")
display(final_df[['file_name', 'category_name']].head(10))

### Step 1.5 – Create the JSONL Training File

The Qwen2-VL model expects data in JSONL (JSON Lines) format, where each line is a JSON object containing an image path and its label. We aggregate all damage categories per image into a comma-separated string.

In [ ]:
import json
import os

output_jsonl_path = 'output_annotations.jsonl'
images_base_path = os.path.join(
    dataset_path,
    'Car Damage Severity Detection-CarDD.v3i.coco',
    'train'
)

# Group by image, aggregate unique category names
grouped_df = (
    final_df
    .groupby('file_name')['category_name']
    .apply(lambda x: ', '.join(x.astype(str).unique()))
    .reset_index()
)

with open(output_jsonl_path, 'w') as f:
    for _, row in grouped_df.iterrows():
        full_image_path = os.path.join(images_base_path, row['file_name'])
        data_entry = {
            "image": full_image_path,
            "text": row['category_name']
        }
        f.write(json.dumps(data_entry) + '\n')

print(f"JSONL file created at: {output_jsonl_path}")
print(f"Total unique images:   {len(grouped_df)}")

Let's verify the first few entries of the JSONL file to make sure the format looks correct.

In [ ]:
with open('output_annotations.jsonl', 'r') as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(line.strip())

### Step 1.6 – Load the Base Model and Processor

We load the pre-trained `Qwen2-VL-2B-Instruct` model using **4-bit quantization** (`BitsAndBytesConfig`) to reduce GPU memory usage — essential for running on Colab's free T4 GPU.

> 💡 **What is 4-bit quantization?** It compresses the model's weights from 32-bit floats to 4-bit integers, roughly halving memory usage with only a small impact on quality. The `bnb_4bit_compute_dtype=torch.float16` setting ensures computations still run at half precision for speed and stability.

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

assert torch.cuda.is_available(), "No GPU detected! Go to Runtime → Change runtime type → T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")

# Diagnose which transformers is actually running
import transformers
print(f"transformers version in use: {transformers.__version__}")
print(f"transformers path: {transformers.__file__}")

# 1. Define 4-bit quantisation configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# 2. Load the base model
# device_map is intentionally omitted — it requires accelerate's device management
# API which conflicts with Colab's pre-installed packages. We load in 4-bit (which
# automatically targets CUDA via bitsandbytes) without any device_map argument.
model_id = "Qwen/Qwen2-VL-2B-Instruct"
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
)

# 3. Load the processor
processor = AutoProcessor.from_pretrained(
    model_id,
    min_pixels=256 * 28 * 28,
    max_pixels=1280 * 28 * 28,
)

print("Qwen2-VL loaded successfully.")
print(f"Model device: {next(model.parameters()).device}")

### Step 1.7 – Prepare the Training Dataset

We load the JSONL file and format each example as a Qwen2-VL **chat conversation**:
- The user provides an image and asks: *"Identify the car damage categories visible in this image."*
- The assistant's expected response is the damage label (e.g., `"scratches, dent"`).

We also perform a clean **train/test split at the source**: we shuffle with a fixed seed and take the first 100 examples for training. The remaining examples (index 100 onwards) are held back for evaluation later.

> 💡 **Why split here?** Splitting from the JSONL file ensures that the test set contains images the model has genuinely never seen during training, preventing data leakage.

In [ ]:
from datasets import load_dataset
from qwen_vl_utils import process_vision_info

# 1. Load full dataset
dataset = load_dataset("json", data_files="output_annotations.jsonl", split="train")

# 2. Shuffle with a fixed seed for reproducibility, then split
shuffled_dataset = dataset.shuffle(seed=42)
train_dataset = shuffled_dataset.select(range(100))      # First 100 for training
# Note: rows 100–300 are reserved for the test set (loaded separately in Part 2)

print(f"Total dataset size: {len(dataset)}")
print(f"Training subset:    {len(train_dataset)} examples")

# 3. Define the formatting function
def format_qwen_chat(example):
    """
    Converts a raw data example into the Qwen2-VL chat format.
    The model learns to respond to the user's question with the damage label.
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": example['image']},
                {"type": "text",  "text": "Identify the car damage categories visible in this image."}
            ]
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": example['text']}
            ]
        }
    ]

    # apply_chat_template formats the conversation into a single string Qwen expects
    text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

    # process_vision_info extracts and pre-processes the image tensors
    image_inputs, video_inputs = process_vision_info(messages)

    return {
        "text_input":  text_input,
        "images":      image_inputs,
        "videos":      video_inputs
    }

# 4. Apply the formatting — IMPORTANT: map only over the 100-row training subset
train_dataset = train_dataset.map(format_qwen_chat, remove_columns=train_dataset.column_names)

print(f"\nSample formatted input (first 150 chars):")
print(train_dataset[0]['text_input'][:150], "...")

### Step 1.8 – Define the Collate Function

The collate function is the "glue" between the dataset and the model. It takes a batch of individual examples and:
1. Pads text sequences to a uniform length within the batch
2. Stacks image tensors into a single batch tensor
3. Creates `labels` by cloning `input_ids` so the model has a training target

> 💡 **Note on label masking:** Ideally, we would mask out the user prompt tokens (setting them to -100) so the model only learns to predict the assistant's response. This is skipped here for simplicity. The model still trains correctly, but may learn to predict the question too, which is inefficient. This is a good topic for class discussion!
>
> **Note on images:** This collator assumes every example contains exactly one image, which is true for this dataset.

In [ ]:
def qwen_collate_fn(examples):
    """
    Collates a list of examples into a padded batch ready for the model.
    Assumes each example contains exactly one image.
    """
    texts  = [x["text_input"] for x in examples]
    # Each example has a list of image tensors; we take the first (and only) image
    images = [x["images"][0] for x in examples]

    # Let the processor handle tokenisation and padding
    batch = processor(
        text=texts,
        images=images,
        padding=True,
        return_tensors="pt"
    )

    # Labels are a copy of input_ids; the loss is computed over these
    # A more advanced version would mask prompt tokens with -100 here
    labels = batch["input_ids"].clone()
    batch["labels"] = labels

    return batch

print("Collate function defined.")

### Step 1.9 – Define the Live Plot Callback

To monitor training in real-time, we define a `LivePlotCallback` that updates a loss curve after every logged step. This gives immediate visual feedback on whether the model is learning.

In [ ]:
from transformers import TrainerCallback
from IPython.display import clear_output
import matplotlib.pyplot as plt

class LivePlotCallback(TrainerCallback):
    """Updates a live training loss plot after each logging step."""

    def __init__(self):
        self.losses = []
        self.steps  = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.losses.append(logs["loss"])
            self.steps.append(state.global_step)

            clear_output(wait=True)

            fig, ax = plt.subplots(figsize=(10, 5))
            ax.plot(self.steps, self.losses, label="Training Loss", color='royalblue')
            ax.set_xlabel("Training Steps")
            ax.set_ylabel("Loss")
            ax.set_title("Real-Time Training Loss Curve")
            ax.legend()
            ax.grid(True)
            plt.tight_layout()
            plt.show()

print("LivePlotCallback defined.")

### Step 1.10 – Configure LoRA and Start Training

We now apply **LoRA (Low-Rank Adaptation)** to the loaded model and begin training.

**What is LoRA?** Instead of updating all ~2B model parameters, LoRA injects small trainable matrices (rank `r=16`) into the attention layers. This reduces trainable parameters by ~99%, making fine-tuning feasible on a single GPU.

**Key training settings:**
- `max_steps=80` — caps training regardless of epoch count, keeping the demo short
- `per_device_train_batch_size=1` with `gradient_accumulation_steps=8` — effectively simulates a batch size of 8 without OOM
- `logging_steps=1` — logs every step so the live plot updates frequently

> ⚠️ **Important:** If you need to re-run this cell, reload the base model first (go back to Step 1.6). Calling `get_peft_model()` twice on the same model will raise an error.

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer

# 1. Configure LoRA adapters
# target_modules specifies which linear layers receive LoRA injection
lora_config = LoraConfig(
    r=16,                   # Rank: higher = more capacity, but more memory
    lora_alpha=16,          # Scaling factor (commonly set equal to r)
    target_modules=[
        "q_proj", "v_proj", "k_proj", "o_proj",   # Attention projections
        "gate_proj", "down_proj", "up_proj"         # MLP projections
    ],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

# Guard against double-wrapping if the cell is accidentally re-run
if not hasattr(model, 'peft_config'):
    model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

# 2. Training arguments
args = TrainingArguments(
    output_dir="qwen_finetuned",
    per_device_train_batch_size=1,   # Must be 1 to avoid OOM on T4
    gradient_accumulation_steps=8,   # Effective batch size = 1 * 8 = 8
    num_train_epochs=1,
    max_steps=80,                    # Override epoch count — stops after 80 steps
    learning_rate=2e-4,
    fp16=True,                       # Mixed precision training for speed
    logging_steps=1,                 # Log every step for the live plot
    report_to="none",                # Disable WandB/TensorBoard
    save_strategy="no",              # No mid-training checkpoints (saves disk)
    remove_unused_columns=False,     # Required: keeps our custom columns in the batch
)

# 3. Initialise the Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=qwen_collate_fn,
    callbacks=[LivePlotCallback()]
)

# 4. Start training
print("Starting training... watch the loss curve below!")
trainer.train()

### Step 1.11 – Save the Fine-Tuned Model

We save the LoRA adapter weights and the processor. The adapters are small files (~100 MB) that capture what the model learned. To restore the fine-tuned model, you simply load these adapters onto the base model — you don't need to save the full 2B-parameter model.

In [ ]:
new_model_name = "qwen_finetuned"

# Save LoRA adapter weights and processor configuration
trainer.model.save_pretrained(new_model_name)
processor.save_pretrained(new_model_name)

print(f"Model adapters saved to: ./{new_model_name}/")
print("Files saved:")
for f in os.listdir(new_model_name):
    print(f"  {f}")

model.eval()
print("\nModel set to evaluation mode.")

### Step 1.12 – Back Up Model to Google Drive

Colab sessions are temporary — any files saved to the instance will be lost when the session ends. We copy our saved model folder to Google Drive for permanent storage.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
import os

source_folder      = "qwen_finetuned"
destination_folder = "/content/drive/MyDrive/FinTech_Model_Backup"

# Remove destination if it already exists (e.g., on re-run)
if os.path.exists(destination_folder):
    shutil.rmtree(destination_folder)

shutil.copytree(source_folder, destination_folder)
print(f"Model successfully backed up to Google Drive at:\n  {destination_folder}")

### Step 1.13 – Zip and Download the Model to Your Local Machine

Run this cell to create a zip archive of the fine-tuned adapter folder and trigger a direct download to your computer. This is useful if you want to share the model via a course portal, email, or GitHub Releases.

> 💡 The zip will be around 100–200 MB. Anyone can upload it to their own Colab session and unzip it — see the instructions added to Step 2.1 for how they load it.

In [ ]:
import shutil
import os
from google.colab import files

source_folder = "qwen_finetuned"
zip_filename  = "qwen_finetuned"   # shutil will append .zip automatically

# Create the zip archive in the current working directory
print(f"Zipping '{source_folder}'...")
shutil.make_archive(zip_filename, 'zip', base_dir=source_folder, root_dir='.')
zip_path = f"{zip_filename}.zip"

size_mb = os.path.getsize(zip_path) / (1024 ** 2)
print(f"Created {zip_path} ({size_mb:.1f} MB)")

# Trigger a browser download to your local machine
print("Starting download...")
files.download(zip_path)

---
---

# 🔁 SECTION BOUNDARY — FINE-TUNING COMPLETE

---

> ✅ **If you have already fine-tuned and saved the model**, you can **skip everything above** and start directly from Part 2 below.
>
> **What the cells above did:**
> - Installed packages and downloaded the `car-damage-severity` dataset
> - Converted annotations to JSONL format
> - Loaded `Qwen2-VL-2B-Instruct` with 4-bit quantisation
> - Fine-tuned it with LoRA for 80 steps on 100 training images
> - Saved the LoRA adapters to `qwen_finetuned/` and backed them up to Google Drive
>
> ⚠️ **Do NOT re-run the training cells** unless you want to repeat the full training process.

---
---

# 🔍 PART 2: Inference
> Load the saved model and evaluate it on unseen test images.
> **If starting fresh in a new session, run Step 2.1 first to reload the model.**

## Access the saved Fine-tuned Model. Download the zip file and upload it to the Colab session.

[Access Fine-tuned Model](https://livewarwickac-my.sharepoint.com/:u:/g/personal/u1875813_live_warwick_ac_uk/IQDv1xwRAzRXR6NHyU2ySzv1Ad_4FoHMfnOoMuuU2aa8ips?e=yfvTJt)

### Step 2.1 – Reload the Fine-Tuned Model (Run This If Starting Fresh)

If you are in a **new Colab session**, choose whichever option applies and set `saved_model_path` accordingly before running the code cell below.

**Option A – You have the zip file** (shared by your instructor via a course portal or download link):
```python
# 1. Upload qwen_finetuned.zip via the Colab file browser (folder icon, left sidebar)
# 2. Unzip it:
import zipfile
with zipfile.ZipFile('qwen_finetuned.zip', 'r') as z:
    z.extractall('.')
saved_model_path = 'qwen_finetuned'
```

**Option B – You have Google Drive access:**
```python
from google.colab import drive
drive.mount('/content/drive')
saved_model_path = '/content/drive/MyDrive/FinTech_Model_Backup'
```

**Option C – You just completed fine-tuning in this same session:**
```python
saved_model_path = 'qwen_finetuned'  # already on disk from Step 1.11
```

> Set `saved_model_path` in the code cell below to match your chosen option, then run it.

In [ ]:
# 1. Upload qwen_finetuned.zip via the Colab file browser (folder icon, left sidebar)
# 2. Unzip it:
import zipfile
with zipfile.ZipFile('qwen_finetuned.zip', 'r') as z:
    z.extractall('.')
saved_model_path = 'qwen_finetuned'

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel # Import PeftModel
import os

assert torch.cuda.is_available(), "No GPU detected! Go to Runtime → Change runtime type → T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── Change this path if loading from Google Drive ──────────────────────────
# saved_model_path = "/content/drive/MyDrive/FinTech_Model_Backup"
saved_model_path = "qwen_finetuned"
# ──────────────────────────────────────────────────────────────────────────

# The previous cell (Step 2.1) should have already unzipped the model if needed.
# We assume saved_model_path now points to the directory containing the adapters.

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# --- CORRECTED MODEL LOADING LOGIC ---
model_id = "Qwen/Qwen2-VL-2B-Instruct" # Original base model ID

# 1. Load the base model first (with quantization)
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    trust_remote_code=True # Added to potentially resolve loading issues
)

# 2. Load the processor for the base model (using the base model's ID)
# Note: If processor config was saved with adapters and differs, use saved_model_path instead.
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True) # Added to potentially resolve loading issues

# 3. Load the LoRA adapters onto the base model
# PeftModel.from_pretrained will correctly load only the adapters
# and attach them to the base_model.
model = PeftModel.from_pretrained(base_model, saved_model_path)

# (Optional but recommended) Merge LoRA weights into the base model for faster inference
# This also converts the model back to a standard transformers model type (not PeftModel)
model = model.merge_and_unload()

model.eval()

print(f"Fine-tuned model loaded from: {saved_model_path}")
print(f"Model device: {next(model.parameters()).device}")
print("Ready for inference.")

### Step 2.2 – Prepare the Test Dataset

We load the full shuffled dataset (same random seed as training) and select rows 100–300 as our **test set** — examples the model has never seen.

> 💡 **Why rows 100–300?** We used rows 0–99 for training. By using the same `shuffle(seed=42)`, we guarantee the split is identical, so there's no overlap between training and test images.

# Run the cells Step 1.3 to Step 1.5 first

In [ ]:
import kagglehub
dataset_path = kagglehub.dataset_download("sujithkumarmanick/car-damage-severity")
print(f"Dataset downloaded to: {dataset_path}")

In [ ]:
import json
import pandas as pd

annotation_file_path = os.path.join(
    dataset_path,
    'Car Damage Severity Detection-CarDD.v3i.coco',
    'train_annotations.coco.json'
)

with open(annotation_file_path, 'r') as f:
    annotations = json.load(f)

print(f"Number of images:      {len(annotations['images'])}")
print(f"Number of annotations: {len(annotations['annotations'])}")
print(f"Number of categories:  {len(annotations['categories'])}")

print("\nFirst 5 annotations:")
display(pd.DataFrame(annotations['annotations']).head())

print("\nCategories:")
display(pd.DataFrame(annotations['categories']))

In [ ]:
import pandas as pd

images_df = pd.DataFrame(annotations['images'])
annotations_df = pd.DataFrame(annotations['annotations'])
categories_df = pd.DataFrame(annotations['categories'])

# Rename to avoid column conflicts on merge
categories_df = categories_df.rename(columns={'id': 'category_id', 'name': 'category_name'})

# Join annotations with categories, then with image filenames
merged_annotations = pd.merge(
    annotations_df,
    categories_df[['category_id', 'category_name']],
    on='category_id',
    how='left'
)
final_df = pd.merge(
    images_df[['id', 'file_name']],
    merged_annotations[['image_id', 'category_name']],
    left_on='id',
    right_on='image_id',
    how='left'
)

print(f"Total annotation rows: {len(final_df)}")
print("\nSample (first 10 rows):")
display(final_df[['file_name', 'category_name']].head(10))

In [ ]:
import json
import os

output_jsonl_path = 'output_annotations.jsonl'
images_base_path = os.path.join(
    dataset_path,
    'Car Damage Severity Detection-CarDD.v3i.coco',
    'train'
)

# Group by image, aggregate unique category names
grouped_df = (
    final_df
    .groupby('file_name')['category_name']
    .apply(lambda x: ', '.join(x.astype(str).unique()))
    .reset_index()
)

with open(output_jsonl_path, 'w') as f:
    for _, row in grouped_df.iterrows():
        full_image_path = os.path.join(images_base_path, row['file_name'])
        data_entry = {
            "image": full_image_path,
            "text": row['category_name']
        }
        f.write(json.dumps(data_entry) + '\n')

print(f"JSONL file created at: {output_jsonl_path}")
print(f"Total unique images:   {len(grouped_df)}")

In [ ]:
from datasets import load_dataset

full_dataset = load_dataset("json", data_files="output_annotations.jsonl", split="train")

# Use the same seed as training to reproduce the exact same shuffle
shuffled_full = full_dataset.shuffle(seed=42)
test_dataset  = shuffled_full.select(range(100, 300))   # Rows 100–300: unseen by model

print(f"Full dataset size: {len(full_dataset)}")
print(f"Test set size:     {len(test_dataset)} examples")

### Step 2.3 – Define the Prediction Function

The `predict_damage` function takes an image path and runs it through the fine-tuned model to get a damage category prediction. It uses `add_generation_prompt=True` so the model knows it's in "response mode".

In [ ]:
from qwen_vl_utils import process_vision_info

def predict_damage(image_path):
    """
    Runs inference on a single image and returns the predicted damage categories.

    Args:
        image_path: Absolute path to the image file.

    Returns:
        Predicted label string (lowercase, stripped).
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text",  "text": "Identify the car damage categories visible in this image."}
            ]
        }
    ]

    # Format the prompt (with generation prompt so model knows to respond)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    # Generate the assistant's response (short label: max 20 new tokens is sufficient)
    generated_ids = model.generate(**inputs, max_new_tokens=20)

    # Trim the input tokens so we only decode the new response tokens
    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    return output_text.strip().lower()   # Normalise for fair comparison

print("predict_damage() defined and ready.")

### Step 2.4 – Run Inference on the Test Set

We loop through all 200 test images, collect predictions and ground-truth labels, and display a few examples.

In [ ]:
from tqdm import tqdm

y_true = []
y_pred = []

print("Running inference on 200 test images...")

for row in tqdm(test_dataset):
    true_label = row['text'].strip().lower()

    try:
        pred_label = predict_damage(row['image'])
    except Exception as e:
        print(f"Error on {row['image']}: {e}")
        pred_label = "error"

    y_true.append(true_label)
    y_pred.append(pred_label)

print("\n--- Sample Predictions (first 5) ---")
for i in range(5):
    print(f"True: {y_true[i]:<30}  |  Pred: {y_pred[i]}")

### Step 2.5 – Inspect Results in a DataFrame

Organising predictions in a DataFrame makes it easy to visually scan for correct predictions, partial matches, and common failure cases.

In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    'True Label':      y_true,
    'Predicted Label': y_pred,
    'Correct':         [t == p for t, p in zip(y_true, y_pred)]
})

print(f"Exact match accuracy: {results_df['Correct'].mean():.1%}")
print(f"\nFirst 10 predictions:")
display(results_df.head(10))

### Step 2.6 – Calculate Accuracy and Discuss Results

We compute overall **exact-match accuracy** using scikit-learn.

> ⚠️ **Important caveat for students:** Exact string matching is a very strict metric for this task. The model might predict `"dent, scratches"` when the true label is `"scratches, dent"` — semantically identical, but counted as wrong. A low accuracy here doesn't necessarily mean the model has failed; it may be predicting the correct categories in a different order, or generating slightly different wording.
>
> This is a great discussion point: how would you design a fairer evaluation metric for this multi-label classification task? (Hints: set-based comparison, F1 score on individual category tokens, etc.)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_true, y_pred)
print(f"Exact-match Accuracy: {accuracy:.2%}")
print()
print("Note: Low accuracy on this metric does not necessarily mean the model is")
print("performing poorly — label order differences count as mismatches.")
print("Consider inspecting the results_df above to assess quality qualitatively.")